# Prompt engineering and answer generation

Run this notebook independently from the top; retrieve.ipynb does not need to be open. It imports the same retrieval functions and loads saved document vectors. Your build_messages, generate_answer, and ask_question functions remain notebook functions. No document embeddings are regenerated.


In [ ]:
from pathlib import Path
import sys
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

PROJECT_ROOT = Path(
    "/Volumes/code/MyProjects/ML/University Handbook Assistant From Scratch"
)
processed_dir = PROJECT_ROOT / "data" / "processed"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.retrieval import (
    load_json, tokenize, tokenize_chunks, hybrid_search,
    rerank_search, prepare_context, add_file_and_prepare_context,
)

from app.generate import Generation


In [ ]:
document_chunks, embeddings, config = load_json(processed_dir)
embedding_model = SentenceTransformer(config["embedding_model"])
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2", max_length=512,
)
tokenized_chunks = tokenize_chunks(document_chunks)
bm25 = BM25Okapi(tokenized_chunks)
print("Retrieval models and BM25 ready.")


## Build the prompt


In [ ]:
from app.retrieval import build_messages


## Load the generator once


In [ ]:
generator = Generation()
print("Generator ready on:", generator.device)


## Generate and connect the pipeline


In [ ]:
def generate_answer(messages, max_tokens=300):
    return generator.generate_answer(messages, max_tokens=max_tokens)


In [ ]:
def ask_question(
    question, top_k=5, candidate_k=30, max_new_tokens=300,
    embeddings=None, document_chunks=None,
):
    if not question.strip():
        raise ValueError("Enter a question.")
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens must be positive.")
    if embeddings is None or document_chunks is None:
        raise ValueError("Pass embeddings and document_chunks.")

    ranked_chunks = rerank_search(
        question, embeddings, document_chunks,
        top_k=top_k, candidate_k=candidate_k,
        embedding_model=embedding_model, reranker=reranker,
    )
    evidence = prepare_context(question, ranked_chunks)
    messages = build_messages(evidence)
    answer = generate_answer(messages, max_tokens=max_new_tokens)
    return {
        "question": question,
        "answer": answer,
        "sources": evidence["results"],
    }


In [ ]:
question = "What are the disadvantages of cloud computing?"
response = ask_question(
    question, embeddings=embeddings, document_chunks=document_chunks,
    top_k=5, candidate_k=30, max_new_tokens=300,
)
print(response["answer"])


## Optional: external PDF or TXT

Run the next cell only when you want to add a file. It embeds only the new file and returns updated arrays. These updates stay in memory. Reusing an already-indexed file adds duplicates. Do not pass the old BM25 index with updated chunks; the helper builds a fresh one.


In [ ]:
# file_path = Path("/absolute/path/to/new_document.pdf")
# evidence, document_chunks, embeddings = add_file_and_prepare_context(
#     file_path=file_path,
#     question=question,
#     document_chunks=document_chunks,
#     embeddings=embeddings,
#     embedding_model=embedding_model,
#     reranker=reranker,
# )
# messages = build_messages(evidence)
# answer = generate_answer(messages, max_tokens=300)
# print(answer)
